# Reaction-path analysis

This optional notebook generates nitrogen reaction-path diagrams for a selected two-stage case. It imports the shared model settings and solver helpers from `staged_combustion.py`; run it from the repository root. Graphviz (`dot`) is required to render the diagrams.


In [ ]:
import shutil
import subprocess
from pathlib import Path

import cantera as ct
from IPython.display import Image, display
from staged_combustion import (
    AIR,
    MECH,
    P_IN,
    T_IN,
    calculate_air_amounts,
    integrate_to_steady_state,
)


In [ ]:
def run_case_for_reaction_paths(
    x_nh3,
    phi_1,
    tau,
    output_prefix="reaction_path"
):
    
    x_h2 = 1.0 - x_nh3

    # ---------------------------------------------------------
    # Calculate air amounts
    # ---------------------------------------------------------
    n_air_stoic, n_air_1, n_air_2 = calculate_air_amounts(
        x_nh3,
        phi_1
    )

    # ---------------------------------------------------------
    # Fuel stream
    # ---------------------------------------------------------
    gas = ct.Solution(MECH)

    fuel_stream = ct.Quantity(gas, constant="HP")
    fuel_stream.TPX = T_IN, P_IN, f"NH3:{x_nh3}, H2:{x_h2}"
    fuel_stream.moles = 1.0

    # ---------------------------------------------------------
    # PRIMARY STAGE
    # ---------------------------------------------------------
    air_stream_1 = ct.Quantity(gas, constant="HP")
    air_stream_1.TPX = T_IN, P_IN, AIR
    air_stream_1.moles = n_air_1

    primary_mix = fuel_stream + air_stream_1

    gas1 = ct.Solution(MECH)
    gas1.TPX = primary_mix.phase.TPX

    # Initial guess
    gas1.equilibrate("HP")

    reactor1 = ct.IdealGasReactor(
        gas1,
        energy="on",
        volume=1.0,
        clone=False
    )

    inlet1 = ct.Reservoir(gas1, clone=False)

    exhaust1 = ct.Reservoir(gas1, clone=False)

    mfc1 = ct.MassFlowController(
        inlet1,
        reactor1,
        mdot=lambda t: reactor1.mass / tau
    )

    pc1 = ct.PressureController(
        reactor1,
        exhaust1,
        primary=mfc1,
        K=0.01
    )

    sim1 = ct.ReactorNet([reactor1])

    converged1, integration_time_1 = integrate_to_steady_state(
        sim1,
        reactor1,
        tau
    )

    if not converged1:
        raise RuntimeError(
            "Primary PSR did not reach steady state."
        )

    print("Primary PSR converged.")
    print(f"Primary temperature = {reactor1.T:.2f} K")

    # ---------------------------------------------------------
    # SAVE PRIMARY OUTLET STATE
    # ---------------------------------------------------------
    primary_path_gas = ct.Solution(MECH)

    primary_path_gas.TDY = reactor1.phase.TDY

    # ---------------------------------------------------------
    # SECONDARY STAGE
    # ---------------------------------------------------------
    primary_products = ct.Quantity(
        primary_path_gas,
        constant="HP"
    )

    primary_products.moles = primary_mix.moles

    air_stream_2 = ct.Quantity(gas, constant="HP")
    air_stream_2.TPX = T_IN, P_IN, AIR
    air_stream_2.moles = n_air_2

    secondary_mix = primary_products + air_stream_2

    overall_phi = n_air_stoic / (n_air_1 + n_air_2)

    gas2 = ct.Solution(MECH)
    gas2.TPX = secondary_mix.phase.TPX

    # Initial guess
    gas2.equilibrate("HP")

    reactor2 = ct.IdealGasReactor(
        gas2,
        energy="on",
        volume=1.0,
        clone=False
    )

    inlet2 = ct.Reservoir(gas2, clone=False)

    exhaust2 = ct.Reservoir(gas2, clone=False)

    mfc2 = ct.MassFlowController(
        inlet2,
        reactor2,
        mdot=lambda t: reactor2.mass / tau
    )

    pc2 = ct.PressureController(
        reactor2,
        exhaust2,
        primary=mfc2,
        K=0.01
    )

    sim2 = ct.ReactorNet([reactor2])

    converged2, integration_time_2 = integrate_to_steady_state(
        sim2,
        reactor2,
        tau
    )

    if not converged2:
        raise RuntimeError(
            "Secondary PSR did not reach steady state."
        )

    print("Secondary PSR converged.")
    print(f"Secondary temperature = {reactor2.T:.2f} K")

    # ---------------------------------------------------------
    # SAVE SECONDARY OUTLET STATE
    # ---------------------------------------------------------
    secondary_path_gas = ct.Solution(MECH)

    secondary_path_gas.TDY = reactor2.phase.TDY

    # ---------------------------------------------------------
    # CREATE REACTION PATH DIAGRAMS
    # ---------------------------------------------------------
    primary_path = create_reaction_path_diagram(
        primary_path_gas,
        element="N",
        title=(
            f"Primary PSR N Reaction Paths\n"
            f"NH3={x_nh3:.2f}, "
            f"phi1={phi_1:.2f}, "
            f"tau={tau*1000:.0f} ms"
        ),
        filename=f"{output_prefix}_primary",
        threshold=0.01,
        label_threshold=0.01
    )

    secondary_path = create_reaction_path_diagram(
        secondary_path_gas,
        element="N",
        title=(
            f"Secondary PSR N Reaction Paths\n"
            f"NH3={x_nh3:.2f}, "
            f"phi1={phi_1:.2f}, "
            f"tau={tau*1000:.0f} ms"
        ),
        filename=f"{output_prefix}_secondary",
        threshold=0.01,
        label_threshold=0.01
    )

    return primary_path, secondary_path

In [ ]:
def create_reaction_path_diagram(
    gas,
    element="N",
    title="Reaction Path Diagram",
    filename="reaction_path",
    threshold=0.0001,
    label_threshold=0.0001,
    flow_type="NetFlow",
    show_details=True
):
    """
    Create a detailed Cantera reaction path diagram.

    threshold:
        Minimum relative flux to display.
        Lower -> more complete pathways.

    label_threshold:
        Minimum relative flux for species labels.

    flow_type:
        "NetFlow"    -> net direction of elemental flow
        "OneWayFlow" -> forward and reverse contributions separately

    show_details:
        Include reaction details contributing to each pathway.
    """

    # ---------------------------------------------------------
    # Check Graphviz
    # ---------------------------------------------------------
    dot_executable = shutil.which("dot")

    if dot_executable is None:
        raise RuntimeError(
            "Graphviz 'dot' executable was not found."
        )

    # ---------------------------------------------------------
    # Create reaction path diagram
    # ---------------------------------------------------------
    diagram = ct.ReactionPathDiagram(
        gas,
        element
    )

    # ---------------------------------------------------------
    # Diagram settings
    # ---------------------------------------------------------
    diagram.title = title

    # Much lower threshold than before
    diagram.threshold = threshold

    # Show labels for weaker pathways too
    diagram.label_threshold = label_threshold

    # Flow direction
    diagram.flow_type = flow_type

    # Show reactions contributing to pathways
    diagram.show_details = show_details

    # Normalize arrow widths using maximum flux
    diagram.scale = -1

    # Make arrows scale with flux
    diagram.arrow_width = -5

    # ---------------------------------------------------------
    # Build
    # ---------------------------------------------------------
    diagram.build()

    # ---------------------------------------------------------
    # Output files
    # ---------------------------------------------------------
    dot_file = Path(f"{filename}.dot")
    png_file = Path(f"{filename}.png")

    diagram.write_dot(str(dot_file))

    # ---------------------------------------------------------
    # DOT -> PNG
    # ---------------------------------------------------------
    subprocess.run(
        [
            dot_executable,
            str(dot_file),
            "-Tpng",
            f"-o{png_file}",
            "-Gdpi=200"
        ],
        check=True
    )

    print(f"Reaction path diagram saved:")
    print(f"  DOT : {dot_file}")
    print(f"  PNG : {png_file}")

    return diagram

In [ ]:
test_case = (
    0.50,   # 50% NH3
    1.50,   # phi1 = 1.5
    0.060   # 60 ms
)

In [ ]:
primary_path, secondary_path = run_case_for_reaction_paths(
    x_nh3=0.50,
    phi_1=1.50,
    tau=0.060,
    output_prefix="NH3_50_phi1_1p5_tau60ms"
)

In [ ]:
from IPython.display import Image, display

display(Image(
    filename="NH3_50_phi1_1p5_tau60ms_primary.png"
))

display(Image(
    filename="NH3_50_phi1_1p5_tau60ms_secondary.png"
))

## 18. Scientific interpretation notes

- **Temperature:** compare primary and final PSR temperatures to see how
  secondary-air addition changes the combustion state.
- **H₂:** final H₂ mole fraction is a fuel-slip / burnout indicator.
- **NH₃:** final NH₃ mole fraction is the main NH₃-slip indicator.
- **NO and NO₂:** report them separately because their trends can differ.
- **NOx:** use `NO + NO₂` as the main NOx metric.
- **N₂O:** keep it separate from NOx, as in the reference presentation.
- **Primary vs final:** primary-stage values describe the first,
  fuel-rich reactor; final values describe the exhaust after secondary
  air addition and the second PSR.
- **Residence time:** the current study assumes the same residence time
  in both PSRs and sweeps 20, 40, 60, 80 and 100 ms.
- **Conversion:** the existing conversion values are preliminary. For
  final scientific reporting, calculate conversion from actual species
  mass-flow rates.
- **Mechanism limitation:** CO/CO₂ should not be added unless a
  carbon-containing mechanism is introduced, because the supplied
  mechanism has no carbon chemistry.

